# Phase 2 — Data Understanding

Companion to PLAN.md Phase 2. Acquires Food-101, runs EDA, audits quality, and locks train/val/test splits. See `docs/superpowers/specs/2026-05-07-phase2-data-understanding-design.md`.

## 0. Setup

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm


In [2]:
# Constants — change deliberately, every downstream phase depends on these.
SEED = 42
VAL_FRACTION = 0.10
PIXEL_STATS_SAMPLE = 2000
TINY_SHORT_SIDE = 256
PHASH_HAMMING_THRESHOLD = 5
PHASH_SCOPE = 'within_class'

FORCE_REBUILD = False        # re-compute cached EDA / audit cells
REGENERATE_SPLITS = False    # re-shuffle splits (DANGEROUS — invalidates downstream comparisons)

random.seed(SEED)
np.random.seed(SEED)


In [3]:
# Paths — relative to repo root. Notebook is launched from repo root.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = REPO_ROOT / 'data'
ARTIFACTS_DIR = REPO_ROOT / 'artifacts' / 'phase2'
FIGURES_DIR = REPO_ROOT / 'figures' / 'phase2'
SPLITS_DIR = REPO_ROOT / 'splits'

for d in [DATA_ROOT, ARTIFACTS_DIR, FIGURES_DIR, FIGURES_DIR / 'samples', SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EDA_STATS_PATH = ARTIFACTS_DIR / 'eda_stats.json'
BAD_FILES_PATH = ARTIFACTS_DIR / 'bad_files.json'
PHASH_CACHE_PATH = ARTIFACTS_DIR / 'phash_cache.parquet'
AUDIT_CACHE_PATH = ARTIFACTS_DIR / 'audit_cache.parquet'

print(f'REPO_ROOT = {REPO_ROOT}')


REPO_ROOT = /Users/nguyenviethung/swin-transformer-classification


In [4]:
def update_stats(key, value):
    """Read eda_stats.json, set one key, write back. Idempotent."""
    stats = {}
    if EDA_STATS_PATH.exists():
        stats = json.loads(EDA_STATS_PATH.read_text())
    stats[key] = value
    EDA_STATS_PATH.write_text(json.dumps(stats, indent=2, sort_keys=True))
    return stats

# Seed eda_stats.json with the constants we already know.
update_stats('split_seed', SEED)
update_stats('imagenet_stats', {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]})


{'split_seed': 42,
 'imagenet_stats': {'mean': [0.485, 0.456, 0.406],
  'std': [0.229, 0.224, 0.225]}}

In [5]:
# ASSERT: setup primitives are defined and paths exist
for name in ['SEED', 'VAL_FRACTION', 'PIXEL_STATS_SAMPLE', 'TINY_SHORT_SIDE',
             'PHASH_HAMMING_THRESHOLD', 'PHASH_SCOPE', 'FORCE_REBUILD',
             'REGENERATE_SPLITS', 'DATA_ROOT', 'ARTIFACTS_DIR', 'FIGURES_DIR',
             'SPLITS_DIR', 'EDA_STATS_PATH', 'BAD_FILES_PATH', 'update_stats']:
    assert name in dir(), f'missing setup symbol: {name}'
for d in [DATA_ROOT, ARTIFACTS_DIR, FIGURES_DIR, SPLITS_DIR]:
    assert d.exists(), f'missing dir: {d}'
print('setup OK')

setup OK
